# 04 – Anomaly Detection (Ensemble + Autoencoder)

Ensemble di **4 modelli** complementari:

| Modello | Tipo | Peso |
|---------|------|------|
| IsolationForest | Alberi casuali — anomalie globali | 35% |
| LocalOutlierFactor | Densità locale — anomalie contestuali | 30% |
| Z-score Baseline | Segnale statistico da notebook 03 | 15% |
| **Autoencoder** | Errore di ricostruzione — pattern non lineari | **20%** |

**Soglie**: derivate dai dati (percentili) invece che fissate manualmente.

**Input**: `features_with_baseline.csv`  
**Output**: `anomaly_results.csv` · `anomaly_summary.json`

In [1]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from sklearn.ensemble import IsolationForest, GradientBoostingClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

ROOT     = Path("..")
PROC_DIR = ROOT / "data" / "processed"
print("Librerie caricate")

Librerie caricate


## 1. Caricamento dati

In [2]:
features_wb = pd.read_csv(PROC_DIR / "features_with_baseline.csv")

ANOMALY_FEATURES = [
    "tot_allarmi_log", "pct_interpol", "pct_sdi", "pct_nsis",
    "tasso_chiusura", "tasso_rilevanza", "tasso_allarme_medio",
    "tasso_inv_medio", "score_rischio_esiti", "tasso_respinti", "tasso_fermati",
    # Nuove feature integrate dal notebook del collega
    "false_positive_rate",
    "alarm_per_invest",
]

print(f"features_with_baseline: {features_wb.shape[0]} rotte x {features_wb.shape[1]} colonne")
print(f"Feature per anomaly detection: {len(ANOMALY_FEATURES)}")
print(f"  {ANOMALY_FEATURES}")

features_with_baseline: 567 rotte x 87 colonne
Feature per anomaly detection: 11
  ['tot_allarmi_log', 'pct_interpol', 'pct_sdi', 'pct_nsis', 'tasso_chiusura', 'tasso_rilevanza', 'tasso_allarme_medio', 'tasso_inv_medio', 'score_rischio_esiti', 'tasso_respinti', 'tasso_fermati']


## 2. Preprocessing — Scaling

In [3]:
X_raw    = features_wb[ANOMALY_FEATURES].fillna(0).copy()
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled_df = pd.DataFrame(X_scaled, columns=ANOMALY_FEATURES, index=features_wb.index)

print(f"Feature matrix X: {X_scaled.shape}")
print(f"Media dopo scaling (deve essere ~0): {[round(float(v),3) for v in X_scaled.mean(axis=0)[:3]]}...")
print(f"Std  dopo scaling (deve essere ~1):  {[round(float(v),3) for v in X_scaled.std(axis=0)[:3]]}...")

Feature matrix X: (567, 11)
Media dopo scaling (deve essere ~0): [0.0, 0.0, 0.0]...
Std  dopo scaling (deve essere ~1):  [1.0, 1.0, 1.0]...


## 3. Modello 1 — Isolation Forest

Isola le anomalie con meno split rispetto ai punti normali.  
`contamination=0.10` assume ~10% di anomalie nel dataset.

In [4]:
iso_forest = IsolationForest(
    n_estimators=200, contamination=0.10,
    max_samples="auto", random_state=42, n_jobs=-1,
)
iso_forest.fit(X_scaled)

if_scores_raw = iso_forest.score_samples(X_scaled)  # piu negativo = piu anomalo
# Normalizza a [0,1]: 1 = massima anomalia
if_max, if_min = if_scores_raw.max(), if_scores_raw.min()
if_anomaly = (if_scores_raw - if_max) / (if_min - if_max)
if_anomaly = np.clip(if_anomaly, 0, 1)

n_if = int((iso_forest.predict(X_scaled) == -1).sum())
print(f"IsolationForest: {n_if} anomalie ({100*n_if/len(X_scaled):.1f}%)")
print(f"  score: min={if_anomaly.min():.4f}, max={if_anomaly.max():.4f}, mean={if_anomaly.mean():.4f}")

IsolationForest: 57 anomalie (10.1%)
  score: min=-0.0000, max=1.0000, mean=0.3014


## 4. Modello 2 — Local Outlier Factor

Confronta la densità locale di ogni punto con quella dei suoi vicini.  
Anomale = bassa densità rispetto ai vicini.

In [5]:
lof = LocalOutlierFactor(
    n_neighbors=20, contamination=0.10,
    metric="euclidean", n_jobs=-1,
)
lof.fit(X_scaled)

lof_scores_raw = -lof.negative_outlier_factor_  # piu alto = piu anomalo
lof_max, lof_min = lof_scores_raw.max(), lof_scores_raw.min()
lof_anomaly = (lof_scores_raw - lof_min) / (lof_max - lof_min)
lof_anomaly = np.clip(lof_anomaly, 0, 1)

n_lof = int((lof.fit_predict(X_scaled) == -1).sum())
print(f"LocalOutlierFactor: {n_lof} anomalie ({100*n_lof/len(X_scaled):.1f}%)")
print(f"  score: min={lof_anomaly.min():.4f}, max={lof_anomaly.max():.4f}, mean={lof_anomaly.mean():.4f}")

LocalOutlierFactor: 57 anomalie (10.1%)
  score: min=0.0000, max=1.0000, mean=0.0238


## 5. Modello 3 — Z-score Baseline

Usa i flag statistici calcolati in `03_baseline_construction.ipynb`.

In [6]:
z_anomaly = features_wb["pct_anomalie"].fillna(0).values

print(f"Z-score baseline:")
print(f"  Rotte con pct_anomalie > 0:     {(z_anomaly > 0).sum()}")
print(f"  Rotte con pct_anomalie >= 0.27: {(z_anomaly >= 0.27).sum()}")
print(f"  z_anomaly: min={z_anomaly.min():.4f}, max={z_anomaly.max():.4f}, mean={z_anomaly.mean():.4f}")

Z-score baseline:
  Rotte con pct_anomalie > 0:     229
  Rotte con pct_anomalie >= 0.27: 11
  z_anomaly: min=0.0000, max=0.3636, mean=0.0473


## 6. Modello 4 — Autoencoder (MLPRegressor)

**Principio**: impara la struttura delle rotte *normali*, poi misura l'**errore di ricostruzione**.  
Rotte anomale = non vengono ricostruite bene → errore alto.

**Architettura**: 11 → 8 → 4 → 8 → 11 (bottleneck a 4 dimensioni)  
**Training**: solo sulle rotte classificate come normali da IsolationForest (first-pass).

In [7]:
# Usa IsolationForest per identificare le rotte 'normali' da usare per il training
normal_mask = iso_forest.predict(X_scaled) == 1
X_normal    = X_scaled[normal_mask]
print(f"Rotte usate per training autoencoder: {X_normal.shape[0]} (normali)")

# Autoencoder tramite MLPRegressor: input = output (task di ricostruzione)
ae = MLPRegressor(
    hidden_layer_sizes=(8, 4, 8),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=1000,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=20,
    verbose=False,
)
ae.fit(X_normal, X_normal)  # input = output

# Errore di ricostruzione (MSE per rotta)
X_reconstructed = ae.predict(X_scaled)
ae_error = np.mean((X_scaled - X_reconstructed) ** 2, axis=1)

# Normalizza a [0,1]
ae_min, ae_max = ae_error.min(), ae_error.max()
ae_anomaly = (ae_error - ae_min) / (ae_max - ae_min)
ae_anomaly = np.clip(ae_anomaly, 0, 1)

n_ae = int((ae_anomaly >= 0.5).sum())
print(f"Autoencoder: {n_ae} rotte con reconstruction error >= 50% del max")
print(f"  Iterazioni training: {ae.n_iter_}")
print(f"  Loss finale:         {ae.loss_:.6f}")
print(f"  ae_anomaly: min={ae_anomaly.min():.4f}, max={ae_anomaly.max():.4f}, mean={ae_anomaly.mean():.4f}")

Rotte usate per training autoencoder: 510 (normali)
Autoencoder: 33 rotte con reconstruction error >= 50% del max
  Iterazioni training: 717
  Loss finale:         0.138940
  ae_anomaly: min=0.0000, max=1.0000, mean=0.1169


## 7. Ensemble Score

Pesi: IF 35% + LOF 30% + Z-score 15% + Autoencoder 20%

Il peso del Autoencoder riflette la sua capacita di catturare pattern non lineari  
che gli altri modelli possono perdere.

In [8]:
W_IF  = 0.35
W_LOF = 0.30
W_Z   = 0.15
W_AE  = 0.20

anomaly_score = (
    W_IF  * if_anomaly  +
    W_LOF * lof_anomaly +
    W_Z   * z_anomaly   +
    W_AE  * ae_anomaly
)

print(f"Ensemble anomaly_score (IF={W_IF} LOF={W_LOF} Z={W_Z} AE={W_AE}):")
print(f"  min={anomaly_score.min():.4f}, max={anomaly_score.max():.4f}, mean={anomaly_score.mean():.4f}")
print(f"  p90={np.percentile(anomaly_score,90):.4f}")
print(f"  p95={np.percentile(anomaly_score,95):.4f}")
print(f"  p97={np.percentile(anomaly_score,97):.4f}")
print(f"  p99={np.percentile(anomaly_score,99):.4f}")

Ensemble anomaly_score (IF=0.35 LOF=0.3 Z=0.15 AE=0.2):
  min=0.0009, max=0.5909, mean=0.1431
  p90=0.2897
  p95=0.3331
  p97=0.3579
  p99=0.4268


## 8. Soglie Data-Driven

Invece di fissare soglie manuali, le deriviamo dalla distribuzione degli score:

- **ALTA**: score >= percentile 97 (top 3% — segnale fortissimo)
- **MEDIA**: score >= percentile 90 (top 10% — segnale significativo)

Questo approccio e difendibile con i dati e adattivo al dataset specifico.

In [9]:
# Soglie derivate dai percentili della distribuzione degli score
ALTA_THRESHOLD  = float(np.percentile(anomaly_score, 97))
MEDIA_THRESHOLD = float(np.percentile(anomaly_score, 90))

# Guardrail: non lasciare che le soglie siano troppo basse
ALTA_THRESHOLD  = max(ALTA_THRESHOLD,  0.30)
MEDIA_THRESHOLD = max(MEDIA_THRESHOLD, 0.20)

print(f"Soglie data-driven:")
print(f"  ALTA  >= {ALTA_THRESHOLD:.4f}  (p97 della distribuzione)")
print(f"  MEDIA >= {MEDIA_THRESHOLD:.4f}  (p90 della distribuzione)")

def classify_anomaly(score):
    if score >= ALTA_THRESHOLD:    return "ALTA"
    elif score >= MEDIA_THRESHOLD: return "MEDIA"
    else:                          return "NORMALE"

labels = np.array([classify_anomaly(s) for s in anomaly_score])
n_alta   = int((labels == "ALTA").sum())
n_media  = int((labels == "MEDIA").sum())
n_norm   = int((labels == "NORMALE").sum())
print()
print("Distribuzione label:")
print(f"  ALTA    : {n_alta:>4} ({100*n_alta/len(labels):.1f}%)")
print(f"  MEDIA   : {n_media:>4} ({100*n_media/len(labels):.1f}%)")
print(f"  NORMALE : {n_norm:>4} ({100*n_norm/len(labels):.1f}%)")

Soglie data-driven:
  ALTA  >= 0.3579  (p97 della distribuzione)
  MEDIA >= 0.2897  (p90 della distribuzione)

Distribuzione label:
  ALTA    :   17 (3.0%)
  MEDIA   :   40 (7.1%)
  NORMALE :  510 (89.9%)


## 9. Risultati e Concordanza tra Modelli

In [10]:
results = features_wb[["ROTTA","PAESE_PART","ZONA","score_composito",
                         "n_anomalie","tot_allarmi_log","pct_interpol",
                         "score_rischio_esiti"]].copy()

results["anomaly_score_if"]  = if_anomaly.round(4)
results["anomaly_score_lof"] = lof_anomaly.round(4)
results["anomaly_score_z"]   = z_anomaly.round(4)
results["anomaly_score_ae"]  = ae_anomaly.round(4)
results["anomaly_score"]     = anomaly_score.round(4)
results["anomaly_label"]     = labels

results["n_models_flagged"] = (
    (if_anomaly  >= 0.5).astype(int) +
    (lof_anomaly >= 0.5).astype(int) +
    (z_anomaly   >= 0.27).astype(int) +
    (ae_anomaly  >= 0.5).astype(int)
)

results = results.sort_values("anomaly_score", ascending=False).reset_index(drop=True)
results["rank"] = results.index + 1

print("Concordanza tra modelli:")
for n in [4, 3, 2, 1]:
    print(f"  Flaggate da almeno {n} modelli: {(results['n_models_flagged'] >= n).sum()}")

print()
print(f"Top 20 rotte anomale:")
cols_show = ["rank","ROTTA","PAESE_PART","anomaly_label","anomaly_score",
             "anomaly_score_if","anomaly_score_lof","anomaly_score_ae",
             "n_models_flagged","tot_allarmi_log","pct_interpol"]
print(results.head(20)[cols_show].to_string(index=False))

Concordanza tra modelli:
  Flaggate da almeno 4 modelli: 0
  Flaggate da almeno 3 modelli: 1
  Flaggate da almeno 2 modelli: 32
  Flaggate da almeno 1 modelli: 102

Top 20 rotte anomale:
 rank   ROTTA            PAESE_PART anomaly_label  anomaly_score  anomaly_score_if  anomaly_score_lof  anomaly_score_ae  n_models_flagged  tot_allarmi_log  pct_interpol
    1 ALG-MXP               Algeria          ALTA         0.5909            1.0000             0.0000            1.0000                 3           5.1874        0.2500
    2 RAK-CIA               Marocco          ALTA         0.5253            0.8700             0.0000            0.9678                 2           3.9512        0.1000
    3 GYD-FCO           Azerbaigian          ALTA         0.4608            0.4210             0.9839            0.0914                 1           1.7918        0.2500
    4 CMN-BLQ               Marocco          ALTA         0.4583            0.8247             0.0000            0.7118                 2

## 10. Quality Check

In [11]:
print("=" * 64)
print("  QUALITY CHECK -- anomaly_results.csv")
print("=" * 64)
print(f"  Rotte totali:    {len(results)}")
print(f"  Null in results: {results.isnull().sum().sum()}")
print(f"  Score range:     [{anomaly_score.min():.4f}, {anomaly_score.max():.4f}]")
print()
print("  Label distribution:")
for lbl in ["ALTA","MEDIA","NORMALE"]:
    n = int((results["anomaly_label"]==lbl).sum())
    print(f"    {lbl:<8}: {n:>4} ({100*n/len(results):.1f}%)")
print()
print("  Top 5 rotte con massimo rischio:")
for _, row in results.head(5).iterrows():
    print(f"    [{int(row['rank']):>3}] {row['ROTTA']:<12} {row['anomaly_label']:<8} score={row['anomaly_score']:.4f}  paese={row['PAESE_PART']}")
print("=" * 64)

  QUALITY CHECK -- anomaly_results.csv
  Rotte totali:    567
  Null in results: 0
  Score range:     [0.0009, 0.5909]

  Label distribution:
    ALTA    :   17 (3.0%)
    MEDIA   :   40 (7.1%)
    NORMALE :  510 (89.9%)

  Top 5 rotte con massimo rischio:
    [  1] ALG-MXP      ALTA     score=0.5909  paese=Algeria
    [  2] RAK-CIA      ALTA     score=0.5253  paese=Marocco
    [  3] GYD-FCO      ALTA     score=0.4608  paese=Azerbaigian
    [  4] CMN-BLQ      ALTA     score=0.4583  paese=Marocco
    [  5] RAK-TSF      ALTA     score=0.4560  paese=Marocco


## 11. Export

In [12]:
out_csv = PROC_DIR / "anomaly_results.csv"
results.to_csv(out_csv, index=False)

summary = {
    "n_routes":         int(len(results)),
    "n_alta":           int((results["anomaly_label"]=="ALTA").sum()),
    "n_media":          int((results["anomaly_label"]=="MEDIA").sum()),
    "n_normale":        int((results["anomaly_label"]=="NORMALE").sum()),
    "alta_threshold":   round(ALTA_THRESHOLD, 4),
    "media_threshold":  round(MEDIA_THRESHOLD, 4),
    "threshold_method": "data-driven (p97/p90)",
    "ensemble_weights": {"IF":W_IF,"LOF":W_LOF,"Z":W_Z,"AE":W_AE},
    "top10_routes": [
        {"rank": int(r["rank"]), "rotta": r["ROTTA"], "label": r["anomaly_label"],
         "score": float(r["anomaly_score"]), "paese": r["PAESE_PART"]}
        for _, r in results.head(10).iterrows()
    ],
}
with open(PROC_DIR / "anomaly_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"anomaly_results.csv  -- {results.shape[0]} rotte, {results.shape[1]} colonne")
print(f"anomaly_summary.json -- {summary['n_alta']} ALTA, {summary['n_media']} MEDIA, {summary['n_normale']} NORMALE")
print(f"Soglie: ALTA>={ALTA_THRESHOLD:.4f}, MEDIA>={MEDIA_THRESHOLD:.4f}")
print(f"Percorso: {out_csv.resolve()}")

anomaly_results.csv  -- 567 rotte, 16 colonne
anomaly_summary.json -- 17 ALTA, 40 MEDIA, 510 NORMALE
Soglie: ALTA>=0.3579, MEDIA>=0.2897
Percorso: /Users/danielegiovanardi/classical-vs-multiagent/classical-vs-multiagent/data/processed/anomaly_results.csv


## 12. Riepilogo

| Modello | Peso | Anomalie rilevate |
|---------|------|-------------------|
| IsolationForest | 35% | Anomalie globali |
| LocalOutlierFactor | 30% | Anomalie locali per densita |
| Z-score Baseline | 15% | Flag statistici multi-feature |
| **Autoencoder** | **20%** | **Pattern non lineari** |

Le soglie ALTA/MEDIA sono derivate automaticamente dalla distribuzione  
degli score ensemble (percentili 97 e 90).

---
*Prossimo step: `05_post_processing.ipynb` — business rules layer*  
*Pipeline Classica — Reply x LUISS 2026*